# Diagnostic de résolution et bruit RAW / T50kpcSUB

Analyse indépendante des modèles et du preprocessing : les WCS et couvertures
proviennent des sorties de validation existantes. Le bruit est mesuré sur les FITS
**natifs avant normalisation**. Aucun FITS ni cache historique n’est réécrit.

Le petit axe de l’ellipse du beam transformée en pixels est classé suivant les
seuils indicatifs `<1`, `[1,2[`, `[2,3[`, `≥3`. Avec des pixels isotropes, cela vaut
BMIN / pixel_scale. Avec anisotropie ou rotation, la covariance du beam est
transformée par la Jacobienne WCS. Les comptes en largeur sont distingués des
comptes en aire dans le CSV et les figures.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Image, Markdown

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "clusterprep/src/clusterprep").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "clusterprep/src"))
from clusterprep.radio_resolution_diagnostics import read_config, run_diagnostic

# Les paramètres persistants se trouvent dans ce JSON. Le modifier pour un
# sous-ensemble (cluster_ids), les seuils de couverture ou overwrite.
CONFIG_PATH = PROJECT_ROOT / "clusterprep/configs/diagnostics/radio_resolution_noise.json"
config = read_config(CONFIG_PATH, PROJECT_ROOT)
config


## Mesures et reprise

Les checkpoints comparent les contenus des FITS natifs, les métadonnées de
validation, les lignes du résumé et le code des fonctions de mesure. Si ces
entrées sont inchangées, les statistiques sont réutilisées ; seules les figures
et le rapport sont régénérés. `overwrite=true` force le recalcul. Un fichier
manquant ajouté ultérieurement est détecté. Ne pas exécuter deux runs dans le
même dossier simultanément.

Le RMS utilise `radio_noise.background_mask` : pixels finis hors rectangle central
de demi-largeur/demi-hauteur, jusqu’à 8 itérations de rejet symétrique à 3σ, avec
σ=1,4826 MAD. Au moins 32 pixels sont requis. C’est une estimation robuste du fond,
avec limites : émission périphérique, ailes de sources, bruit non uniforme et
résidus de soustraction. Les fonds natifs des deux produits peuvent couvrir des
régions différentes. Aucun nombre de pixels n’est assimilé à un nombre de bruits
indépendants.


In [ ]:
metrics, pairs, execution = run_diagnostic(config)
OUTPUT_DIR = Path(config["output_dir"])
print(execution)
print("Rapport :", OUTPUT_DIR / "report.md")
display(metrics.groupby("modality").agg(
    lignes=("cluster_id", "size"), sources=("source_exists", "sum"),
    beams_valides=("beam_valid", "sum"), rms_valides=("noise_valid", "sum"),
))
display(pd.crosstab(metrics.modality, metrics.sampling_class))
display(metrics.groupby("modality")[["beam_minor_arcsec", "pixels_per_beam_minor", "beams_area", "rms_native"]].describe())


## Distributions

Les RMS sont séparés par unité canonique. Un rapport en Jy/beam ne corrige pas les
différences d’aire de beam et ne mesure pas l’effet isolé du tapering : la
soustraction des sources compactes et les résidus interviennent également.
Les lignes 1/2/3 du scatter plot supposent des pixels isotropes ; la classification
CSV utilise la covariance complète. Le nombre de beams en aire est nominal au
beam natif, pas un nombre de détails réellement conservés dans une image 128².


In [ ]:
for name in ["01_pixels_per_beam", "02_pixel_scales", "03_scale_vs_bmin",
             "04_beam_counts", "05_native_noise", "06_coverage"]:
    display(Image(filename=str(OUTPUT_DIR / "figures" / f"{name}.png"), width=1000))


## Cas discordants, couverture et données manquantes

Aucun amas n’est exclu automatiquement. Une couverture absente n’est pas remplacée
par zéro. Une paire exploitable pour le diagnostic possède deux beams valides,
deux RMS mesurables et des unités compatibles ; ce critère ne garantit ni une
couverture suffisante ni une résolution adaptée à l’entraînement.


In [ ]:
display(pairs[pairs.raw_under2_tapered_ge2 | pairs.tapered_under2_raw_ge2])
display(pairs[pairs.raw_low_coverage | pairs.tapered_low_coverage | pairs.large_coverage_difference])
display(pd.read_csv(OUTPUT_DIR / "metadata_issues.csv"))


## Galerie représentative

Les exemples sont déterministes : meilleur petit axe parmi les cas ≥3 pixels
(sinon meilleur disponible, annoncé explicitement), plus petit axe sévèrement
sous-échantillonné et plus faible couverture. Les deux radios sont montrées
côte à côte quand elles existent. Les images sont celles de la validation déjà
sauvegardée, avec la normalisation existante P30/P99/asinh(10), par image ; le gris
signale une absence de couverture. Elles ne montrent aucune correction du beam.


In [ ]:
gallery = json.loads((OUTPUT_DIR / "gallery_selection.json").read_text())
for example in gallery:
    print(example["category"], example["cluster_id"])
    if "figure" in example:
        display(Image(filename=str(OUTPUT_DIR / "figures" / example["figure"]), width=1000))


## Conclusions

Le rapport généré [report.md](../../outputs/radio_resolution_diagnostic/report.md)
contient les effectifs, quantiles, limites et la comparaison A/B/C, ainsi que les
listes complètes des cas à examiner. Les valeurs sont recalculées à partir des
métriques de ce run, pas reprises d’une ancienne pipeline.

A : grille physique commune bilinéaire. B : nombre de beams cible, avec un champ
potentiellement différent selon la modalité. C : grille commune et anti-aliasing.
La recommandation doit distinguer une comparaison à PSF effective commune
(isoler le produit/soustraction) d’une comparaison des produits à résolutions
natives différentes. Aucune stratégie n’est implémentée par ce diagnostic.
